# Level 3 Task 2: Dashboard Preparation

This notebook prepares dashboard-ready churn data and summary tables for Power BI or Tableau.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
base_dir = Path.cwd()
if not (base_dir / "data" / "churn_train.csv").exists():
    base_dir = Path("Level_3_Advanced") / "Task_2_Dashboarding"

train_path = base_dir / "data" / "churn_train.csv"
test_path = base_dir / "data" / "churn_test.csv"
output_dir = base_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
train_df = pd.read_csv(train_path).assign(dataset="train")
test_df = pd.read_csv(test_path).assign(dataset="test")
df = pd.concat([train_df, test_df], ignore_index=True)
df["Churn Label"] = df["Churn"].map({False: "No", True: "Yes"})
df["International Plan Flag"] = df["International plan"].map({"No": "No International Plan", "Yes": "International Plan"})
df["Voice Mail Flag"] = df["Voice mail plan"].map({"No": "No Voice Mail", "Yes": "Voice Mail"})
df["Customer Service Calls Band"] = pd.cut(df["Customer service calls"], bins=[-1, 1, 3, 10], labels=["0-1 Calls", "2-3 Calls", "4+ Calls"])
df["Total Charges"] = (df["Total day charge"] + df["Total eve charge"] + df["Total night charge"] + df["Total intl charge"]).round(2)
df.head()

## KPI Summary

In [ ]:
kpi_df = pd.DataFrame({
    "metric": ["Total Customers", "Churned Customers", "Churn Rate", "Average Total Charges", "Average Customer Service Calls"],
    "value": [
        len(df),
        int(df["Churn"].sum()),
        round(df["Churn"].mean(), 4),
        round(df["Total Charges"].mean(), 2),
        round(df["Customer service calls"].mean(), 2),
    ],
})
kpi_df

## Summary Tables

In [ ]:
state_summary = df.groupby("State").agg(customers=("Churn", "size"), churned_customers=("Churn", "sum"), churn_rate=("Churn", "mean"), avg_total_charge=("Total Charges", "mean")).sort_values(by=["churn_rate", "customers"], ascending=[False, False]).round({"churn_rate": 4, "avg_total_charge": 2}).reset_index()
plan_summary = df.groupby(["International Plan Flag", "Voice Mail Flag"]).agg(customers=("Churn", "size"), churned_customers=("Churn", "sum"), churn_rate=("Churn", "mean")).round({"churn_rate": 4}).reset_index()
service_call_summary = df.groupby("Customer Service Calls Band", observed=False).agg(customers=("Churn", "size"), churned_customers=("Churn", "sum"), churn_rate=("Churn", "mean")).round({"churn_rate": 4}).reset_index()
state_summary.head()

## Dashboard Design Charts

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=state_summary.head(10), x="State", y="churn_rate", color="#e74c3c")
plt.title("Top 10 States by Churn Rate", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=plan_summary, x="International Plan Flag", y="churn_rate", hue="Voice Mail Flag", palette="Set2")
plt.title("Churn Rate by Plan Type", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=service_call_summary, x="Customer Service Calls Band", y="churn_rate", color="#3498db")
plt.title("Churn Rate by Customer Service Calls Band", fontweight="bold")
plt.tight_layout()
plt.show()